# Avance 5 - Modelo final: mejora del mejor modelo (Equipo 17, AgroSatCopilot)

## Proyecto Integrador MNA - Tec de Monterrey

**Equipo 17**

- Carlos Isaac Avila Gutierrez - A01796035
- Carlos Aaron Bocanegra Buitron - A01796345
- Arthur Jafed Zizumbo Velasco - A01796363

**Curso**: MNA - Tec de Monterrey - 20-abr -> 3-jul-2026

**Sponsor academico**: Dr. Gerardo Jose Camacho - gjcamacho@tec.mx

**Fecha de entrega**: 2026-06-07.

---

## Resumen

El Avance 4 evaluo seis arquitecturas de segmentacion densa sobre PASTIS-R. TSViT-pheno obtuvo el mejor desempeno individual: mIoU 0.625, F1-macro 0.750 y pixel-accuracy 0.876 sobre el fold de validacion. Este cuaderno optimiza dicho modelo mediante ponderacion de clases (effective-number) y aumento de datos geometrico, para reducir la diferencia entre la pixel-accuracy (0.876) y el F1-macro (0.750), asociada al desbalance de clases.

Documento de planeacion: docs/us-planning/avance5-mejora-modelo-final.md.

## Objetivos

- Optimizar el modelo individual seleccionado (TSViT-pheno) mediante ponderacion de clases y aumento de datos.
- Comparar el modelo optimizado contra el baseline del Avance 4.
- Analizar el error por clase e interpretar las graficas de desempeno.

Los umbrales de referencia para produccion son F1-macro >= 0.80 y mIoU >= 0.70 sobre las 18 clases. El modelo individual no alcanza dichos umbrales en ese esquema; el esquema agrupado de 6 grupos HCAT se reporta como metrica complementaria y los ensambles se desarrollan en un cuaderno aparte.

## Requisitos de ejecucion

- Entorno con GPU (L4 o superior).
- En Colab, la celda de inicializacion monta el Drive compartido, clona el repositorio (branch `user/abocanegra/semana-5`) e instala las dependencias.
- Los datos PASTIS-R y los artefactos residen en el Drive compartido (`MyDrive/Integrador/`).
- El entrenamiento se ejecuta con `RUN_TRAINING = True`. El checkpoint se persiste en Drive y la corrida admite reanudacion.

In [ ]:
# --- Bootstrap Colab: monta Drive, clona el repo, instala deps ---
import os, subprocess, sys
from pathlib import Path

_IN_COLAB = False
shared_folder_path = ''
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shared_folder_path = '/content/drive/MyDrive/Integrador/'
    _IN_COLAB = True
except ImportError:
    pass

# En Colab el repo no esta presente: se clona una vez.
if _IN_COLAB:
    from getpass import getpass
    _repo_dir = '/content/agrosat-copilot'
    _branch = 'user/abocanegra/semana-5'  # branch con la mejora del modelo
    _repo = 'github.com/ArthurZizumbo/agrosat-copilot.git'
    if not Path(_repo_dir, 'pyproject.toml').is_file():
        _rc = os.system(
            f'git clone --branch {_branch} --depth 1 '
            f'https://{_repo} {_repo_dir}')
        if _rc != 0:  # repo privado: pide token (no se guarda)
            _tok = getpass('GitHub token (repo privado): ')
            os.system(
                f'git clone --branch {_branch} --depth 1 '
                f'https://{_tok}@{_repo} {_repo_dir}')

# Localiza el repo por su pyproject.toml y entra en el.
_search = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
if _IN_COLAB:
    _search = [Path('/content/agrosat-copilot'), *_search]
for _cand in _search:
    if (_cand / 'pyproject.toml').is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        os.chdir(_cand)
        break
else:
    raise RuntimeError('No se encontro el repo agrosat-copilot.')

if _IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                    'segmentation-models-pytorch', 'structlog', 'typer',
                    'polars', 'mlflow', 'optuna'], check=False)

print('repo:', Path.cwd(), '| colab:', _IN_COLAB,
      '| drive:', shared_folder_path or '(local)')

In [ ]:
# --- Configuracion de la corrida (datos y artefactos en Drive) ---
import matplotlib.pyplot as plt  # noqa: F401
import polars as pl

_base = shared_folder_path if shared_folder_path else ''
PASTIS_ROOT = Path(_base + 'data/PASTIS-R')         # datos en Drive
A4_METRICS = Path('reports/segmentation/metrics')   # parquets Avance 4 (git)
STAGE_DIR = Path(_base + 'reports/best_model')       # artefactos Avance 5
METRICS_DIR = STAGE_DIR / 'metrics'
FIGURES_DIR = STAGE_DIR / 'figures'
CHECKPOINT_DIR = STAGE_DIR / 'checkpoints'
for _d in (METRICS_DIR, FIGURES_DIR, CHECKPOINT_DIR):
    _d.mkdir(parents=True, exist_ok=True)
MLFLOW_URI = 'file:' + str(STAGE_DIR / 'mlruns')
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'  # MLflow 3.x file store

RUN_NAME = 'alt-tsvit-pheno-cw-aug-v1'  # variante mejorada
BASE_RUN_NAME = 'alt-tsvit-pheno-v1'    # baseline del Avance 4
EPOCHS = 40                             # best del Avance 4 fue 28/30
DEVICE = 'cuda' if _IN_COLAB else 'auto'
RUN_TRAINING = False                    # True para entrenar (horas)

print('PASTIS_ROOT:', PASTIS_ROOT, '| existe:', PASTIS_ROOT.exists())
print('artefactos Avance 5:', STAGE_DIR)
print('device     :', DEVICE, '| run:', RUN_NAME)

## 1. Modelo de partida (Avance 4)

Tabla comparativa de las arquitecturas del Avance 4 y diferencia respecto a los umbrales de produccion (F1-macro >= 0.80, mIoU >= 0.70).

In [ ]:
# --- Recap Avance 4: tabla comparativa + brecha del mejor modelo ---
parts = sorted(A4_METRICS.glob('model_comparison_avance4_*.parquet'))
# diagonal_relaxed tolera parquets con distinto esquema entre integrantes.
_keep = ['model', 'miou', 'f1_macro', 'pixel_accuracy',
         'miou_grouped', 'f1_macro_grouped', 'pixel_accuracy_grouped']
_frames = []
for _p in parts:
    _d = pl.read_parquet(_p)
    _frames.append(_d.select([c for c in _keep if c in _d.columns]))
if _frames:
    a4 = (pl.concat(_frames, how='diagonal_relaxed')
            .unique(subset=['model'], keep='last')
            .sort('miou', descending=True, nulls_last=True))
    display(a4)
    best = a4.row(0, named=True)
    print(f"Mejor: {best['model']} | mIoU {best['miou']:.3f} "
          f"| F1-macro {best['f1_macro']:.3f}")
    print(f"Brecha -> F1: {0.80 - best['f1_macro']:+.3f} "
          f"| mIoU: {0.70 - best['miou']:+.3f}")
else:
    print('No hay parquets de Avance 4 en', A4_METRICS)

## 2. Estrategia de optimizacion

Se mantiene la arquitectura TSViT-pheno y se aplican tres ajustes orientados al F1-macro de las clases minoritarias:

| Ajuste | Descripcion | Parametro |
|--------|-------------|-----------|
| Ponderacion de clases | CrossEntropy ponderado (effective-number) | `--class-balance effective` |
| Aumento de datos | Flips y rotaciones D4 sincronizados imagen-mascara | `--augment` |
| Entrenamiento extendido | 40 epocas con early-stopping | `--epochs 40 --patience 8` |

Los pesos por clase se calculan sobre los folds de entrenamiento. Los artefactos reutilizables se persisten en `reports/best_model/`.

In [ ]:
# --- Lanzar el entrenamiento de la variante mejorada (subprocess) ---
cmd = [
    sys.executable, '-m', 'ml.train.train_segmentation',
    '--model', 'tsvit-pheno',
    '--target', 'semantic18',
    '--epochs', str(EPOCHS),
    '--patience', '8',
    '--batch-size', '16',
    '--n-timesteps', '10',
    '--device', DEVICE,
    '--root', str(PASTIS_ROOT),
    '--ckpt-dir', str(CHECKPOINT_DIR / RUN_NAME),
    '--mlflow-uri', MLFLOW_URI,
    '--augment',
    '--class-balance', 'effective',
    '--class-balance-beta', '0.9999',
    '--run-name', RUN_NAME,
]
print('comando:', ' '.join(cmd))
if RUN_TRAINING:
    # Stream stdout+stderr en vivo (subprocess.run no se ve en la celda).
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for _line in proc.stdout:
        print(_line, end='')
    proc.wait()
    print('returncode:', proc.returncode)
else:
    print('RUN_TRAINING=False -> no entrena. Pon True para entrenar.')

## 3. Comparativa baseline vs optimizado

Metricas de validacion (fold 4) del baseline del Avance 4 y del modelo optimizado. La seleccion del modelo final se realiza por mIoU.

In [ ]:
# --- Comparativa baseline (Avance 4) vs mejorado (stage Avance 5) ---
rows = []
# Baseline tsvit-pheno: del parquet del Avance 4 (entregable anterior).
_a4 = A4_METRICS / 'model_comparison_avance4_tsvit.parquet'
if _a4.exists():
    _b = pl.read_parquet(_a4).filter(pl.col('model') == 'tsvit-pheno')
    if _b.height:
        _r = _b.row(0, named=True)
        rows.append({'run': BASE_RUN_NAME, 'miou': _r['miou'],
                     'f1_macro': _r['f1_macro'],
                     'pixel_acc': _r['pixel_accuracy']})
# Run mejorado: del MLflow de este stage (reports/best_model/mlruns).
try:
    import mlflow
    os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE', 'true')
    mlflow.set_tracking_uri(MLFLOW_URI)
    df = mlflow.search_runs(search_all_experiments=True)
    name_col = 'tags.mlflow.runName'
    if name_col in df.columns:
        for _, r in df[df[name_col] == RUN_NAME].iterrows():
            rows.append({
                'run': RUN_NAME,
                'miou': r.get('metrics.best_miou', r.get('metrics.miou')),
                'f1_macro': r.get('metrics.best_f1_macro',
                                  r.get('metrics.f1_macro')),
                'pixel_acc': r.get('metrics.best_pixel_acc',
                                   r.get('metrics.pixel_acc')),
            })
except Exception as exc:  # noqa: BLE001
    print('MLflow no disponible:', exc)

if rows:
    comp = pl.DataFrame(rows).sort('miou', descending=True, nulls_last=True)
    display(comp)
    _out = METRICS_DIR / 'avance5_best_model_comparison.parquet'
    comp.write_parquet(str(_out))
    print('escrito', _out)
else:
    print('Pendiente: corre el entrenamiento (RUN_TRAINING=True).')

## 4. Analisis de error por clase

Precision, recall, F1 e IoU por clase sobre el fold de validacion, calculados con el checkpoint del modelo final. Identifica las clases con menor desempeno.

In [ ]:
# --- Tabla per-clase del modelo final (degrada si falta ckpt/datos) ---
import torch
from torch.utils.data import DataLoader

from ml.data.pastis_seg_dataset import PASTISSegmentationDataset
from ml.eval.dense_metrics import DenseConfusionAccumulator

CKPT = CHECKPOINT_DIR / RUN_NAME / 'best.pt'
try:
    if not CKPT.exists():
        raise FileNotFoundError(CKPT)
    from ml.models.tsvit_wrapper import build_tsvit
    model = build_tsvit(num_classes=18, n_timesteps=10, img_size=128,
                        in_channels=10, semantic_dim=384)
    state = torch.load(CKPT, map_location='cpu')
    model.load_state_dict(state.get('model', state))
    dev = 'cuda' if torch.cuda.is_available() and DEVICE != 'cpu' else 'cpu'
    model = model.to(dev).eval()
    val_ds = PASTISSegmentationDataset(
        root=PASTIS_ROOT, folds=(4,), collapse_time=None,
        n_timesteps=10, target='semantic18')
    acc = DenseConfusionAccumulator(num_classes=18, ignore_index=255)
    with torch.no_grad():
        for x, y in DataLoader(val_ds, batch_size=4):
            out = model(x.to(dev))
            logits = out[0] if isinstance(out, tuple) else out
            acc.update(logits.argmax(1).cpu(), y)
    per_class = pl.DataFrame(acc.per_class_metrics()).sort('f1')
    display(per_class)
    per_class.write_parquet(
        str(METRICS_DIR / 'avance5_per_class_tsvit_pheno.parquet'))
except Exception as exc:  # noqa: BLE001
    print('Pendiente (corre el entrenamiento primero):', exc)

## 5. Graficas de desempeno

Curvas de entrenamiento, IoU por clase, matriz de confusion y comparativa, generadas a partir de los artefactos del modelo final.

In [ ]:
# --- Galeria de figuras del modelo final (degrada con placeholder) ---
from IPython.display import Image, Markdown, display

_fig_types = [('curves', 'Curvas de entrenamiento'),
              ('per_class_iou', 'IoU por clase'),
              ('confusion', 'Matriz de confusion'),
              ('samples', 'RGB / verdad / prediccion')]
_models = ('tsvit-pheno-cw-aug', 'tsvit-pheno', 'tsvit_pheno')
_shown = False
for _key, _label in _fig_types:
    for _model in _models:
        _f = FIGURES_DIR / f'{_key}_{_model}.png'
        if _f.exists():
            display(Markdown(f'**{_label}** ({_model})'))
            display(Image(filename=str(_f)))
            _shown = True
            break
if not _shown:
    display(Markdown('_Pendiente: genera las figuras tras el entrenamiento._'))

## 6. Modelo final y evaluacion para produccion

Modelo seleccionado: TSViT-pheno con ponderacion de clases y aumento de datos (`alt-tsvit-pheno-cw-aug-v1`).

La evaluacion para produccion contrasta el desempeno contra los umbrales F1-macro >= 0.80 y mIoU >= 0.70, en los esquemas de 18 clases y de 6 grupos HCAT, con el baseline XGBoost+AlphaEarth como capa de respaldo. La inferencia se ejecuta de forma asincrona.

_Conclusiones a completar tras la ejecucion del entrenamiento._